# Zadanie 1, Część II – Text Embeddings (2 pkt) · PolEmo 2.0

Notebook odpowiada punktom z instrukcji (Text Embeddings, 2 pkt):

| Pkt zadania | Punkty | Sekcja w notebooku |
|---|---|---|
| 1. Osadzenia zdań, 2 różne modele (fastText / uśrednione Word2Vec / USE) | 0,5 | **Sekcja 1** |
| 2. t-SNE z odniesieniem do etykiet, wykresy interaktywne (hover: zdanie i anotacja) | 0,5 | **Sekcja 2** |
| 3. Analiza osadzeń i dyskusja (pytania pomocnicze 1–5) | 1,0 | **Sekcja 3** (3.1–3.5 materiał, 3.6 odpowiedzi) |

Kod - `src/text_embeddings.py`. W notebooku jest tylko uruchomienie i komentarz.
Etykiety do kolorowania: `gold` (PolEmo), `A1`, `A2` (niezależne anotacje obu iteracji) oraz `final`
(uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora, z `annotation/iterN/adjudicated.csv`).

**Uśrednione Word2Vec (pkt 1b).** Dwa modele Word2Vec z części Word Embeddings (IPI PAN i PolEmo) mają osobne podsekcje: **1.4** (osadzenia), **2.1** (t-SNE) i **3.7** (analiza i odpowiedzi na pytania 1–5).

In [ ]:
import os, sys, warnings, logging, contextlib, io
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")
for _n in ("huggingface_hub", "tensorflow", "absl"):
    logging.getLogger(_n).setLevel(logging.ERROR)
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from src import text_embeddings as te

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
# Wykresy interaktywne pod komórkami: „plotly_mimetype” rysują JupyterLab, Notebook 7 i VS Code (także w niezaufanym
# notebooku), „notebook_connected” – klasyczny Notebook i eksport do HTML (plotly.js z CDN, potrzebny internet).
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Konfiguracja

In [ ]:
MODELS        = ["fasttext", "use"]
ITERATIONS    = [1, 2]                # próbki z obu iteracji (po 100 zdań)
INCLUDE_BASELINE = True               # TF-IDF + SVD jako punkt odniesienia (nie liczy się do 2 modeli)
FORCE         = False                 # True = przelicz osadzenia mimo cache

FASTTEXT_PATH = None                  # None = pobierze z Hugging Face (clarin-pl/fastText-kgr10); lub ścieżka do .bin
W2V_PATH      = None                  # ścieżka do modelu z https://dsmodels.nlp.ipipan.waw.pl (wymagana dla "w2v")
USE_BACKEND   = "auto"                # "tfhub" (oryginalny USE), "distiluse" (destylat mUSE) lub "auto"

## 1. Osadzenia zdań [0,5 pkt]

**Dane:** zdania z próbek obu iteracji anotacji (`annotation/iter1`, `annotation/iter2`), etykieta `gold` z PolEmo,
anotacje A1 i A2 z obu iteracji (niezależne, przed rozstrzygnięciem konfliktów przez superanotatora;
zdania bez etykiety dostają w kolorowaniu „brak”). Etykieta `final` to uzgodniona wersja po rozstrzygnięciu konfliktów przez superanotatora.

**Modele** (każdy daje jeden wektor na zdanie):
* **fastText (kgr10)** – średnia wektorów słów zdania; nieznane słowa mają wektor z n-gramów znakowych, więc
  nie ma problemu OOV (ważne przy artefaktach tokenizacji korpusu, np. „wymagał em”).
* **Word2Vec (IPI PAN)** – średnia wektorów słów; słowa spoza słownika są pomijane (raportujemy pokrycie).
* **Universal Sentence Encoder (multilingual)** – wektor całego zdania, bez uśredniania słów.
* **TF-IDF + SVD** – punkt odniesienia (worek słów), niczego nie pobiera.

In [ ]:
df = te.load_sentences(ITERATIONS)
print(len(df), "zdań;", "kolumny etykiet:", te.label_columns(df))
display(df.groupby("iter")["gold"].value_counts().unstack().assign(razem=lambda x: x.sum(axis=1)))
display(df[["sent_id", "iter", "text", "gold", "A1", "A2"]].head(5))

**Kontrola etykiet `final`.** Poniżej liczba zdań z rozstrzygnięciem i tabela: etykieta gold z PolEmo (wiersze) kontra
uzgodniona etykieta `final` (kolumny). Wartości poza przekątną to zdania, w których zespół uznał inną etykietę niż korpus.

In [ ]:
if "final" in te.label_columns(df):
    print("zdań z etykietą final:", int(df["final"].notna().sum()), "z", len(df))
    display(pd.crosstab(df["gold"], df["final"].fillna("brak"), rownames=["gold"], colnames=["final"], margins=True))
    ok = df["final"].notna()
    print(f"zgodność final z gold: {(df.loc[ok, 'final'] == df.loc[ok, 'gold']).mean():.1%}")
else:
    print("UWAGA: brak etykiet final – sprawdź annotation/iterN/adjudicated.csv (kolumny sent_id, final)")

**Jak czytać tabelę.** Wiersze to etykieta z korpusu PolEmo (`gold`), kolumny to etykieta po rozstrzygnięciu konfliktów (`final`), dla wszystkich 200 zdań z obu iteracji. Przekątna to zdania, w których korpus i zespół się zgadzają. Poza przekątną są zdania, którym zespół nadał inną etykietę niż korpus. Największa zmiana to `gold` AMB → `final` NEU, bo wg wytycznych AMB dotyczy sprzecznych ocen w jednym zdaniu, a nie ogólnego wydźwięku recenzji. To porównanie z korpusem, a nie miara zgodności anotatorów (tę liczy `python -m src.agreement`).

In [ ]:
names = MODELS + (["tfidf"] if INCLUDE_BASELINE else [])
embedders, E, failed = {}, {}, {}
for n in names:
    try:
        m = te.get_embedder(n, FASTTEXT_PATH, W2V_PATH, USE_BACKEND)
        E[n] = te.get_embeddings(m, df, force=FORCE)
        embedders[n] = m
        cov = getattr(m, "last_coverage", None)
        extra = f", pokrycie słownika: {cov.mean():.1%}" if cov is not None else ""
        print(f"OK  {n:9s} {E[n].shape}  {m.label}{extra}")
    except Exception as e:
        failed[n] = f"{type(e).__name__}: {e}"
        print(f"BŁĄD {n}: {failed[n]}")
real = [n for n in E if n != "tfidf"]
if len(real) < 2:
    warnings.warn(f"Zadanie wymaga 2 modeli, działają: {real}. Sprawdź błędy powyżej.")

### 1.3 Jak wyglądają osadzenia

Każde zdanie jest teraz **wektorem liczb**: fastText i TF-IDF dają 100 liczb, USE 512. Poniżej fragment macierzy
(pierwsze 5 zdań, pierwsze 8 wymiarów) oraz norma całego wektora. Pojedyncze wymiary nie mają interpretacji,
liczy się dopiero odległość między wektorami. Pełne osadzenia zapisują się do `results/text_embeddings/embeddings_<model>.csv`.

In [ ]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    print(f"macierz: {E[n].shape[0]} zdań × {E[n].shape[1]} wymiarów, dtype {E[n].dtype}, "
          f"średnia norma wektora: {np.linalg.norm(E[n], axis=1).mean():.2f}")
    display(te.embedding_preview(df, E[n], n_rows=5, n_dims=8))
    print("zapisano:", te.export_embeddings_csv(df, E[n], n))

**Podobieństwo kosinusowe wybranych zdań.** Po dwa zdania z każdej klasy gold (iteracja 1). Jasne pola = zdania bliskie
w danej przestrzeni, na przekątnej zawsze 1. w fastText wszystkie zdania są do siebie podobne,
w USE i TF-IDF większość par jest bliska zera.

In [ ]:
sel = (df[df["iter"] == df["iter"].min()].groupby("gold", sort=False).head(2)).sort_values("gold").index
for n in E:
    te.similarity_heatmap(df, E[n], sel, f"Podobieństwo kosinusowe – {embedders[n].label}").show()

### 1.4 Uśrednione wektory Word2Vec (pkt 1b)

> W `Text_Embeddings_executed.ipynb` komórki Word2Vec (1.4, 2.1 i 3.7) wykonano osobno, lokalnie 21.10.2026
> (`python -m src.notebook_w2v`). Pozostałe komórki i ich wyniki pochodzą z uruchomienia zespołu (USE z TF Hub)
> i nie były zmieniane.

Wektor zdania to **średnia wektorów jego słów** (małe litery, bez interpunkcji). Słowa bez wektora są pomijane,
dlatego podajemy pokrycie słownika. Dwa modele z części Word Embeddings, oba 100-wymiarowe:
* **Word2Vec IPI PAN** (`w2v`) – ogólny model z NKJP i Wikipedii (formy słów, skip-gram), 300 tys. najczęstszych form;
* **Word2Vec PolEmo** (`w2v_polemo`) – model wytrenowany w gensim na zdaniach PolEmo (recenzje, model dziedzinowy).

Osadzenia wczytują się z cache `results/text_embeddings/emb_<model>.npy`. Z tych samych osadzeń pochodzą liczby
w sprawozdaniu (podrozdział „Uśrednione wektory Word2Vec”).

In [ ]:
W2V_MODELS = ["w2v", "w2v_polemo"]
W2V_TITLES = {"w2v": "Word2Vec IPI PAN", "w2v_polemo": "Word2Vec PolEmo"}
emb_w2v, E_w2v, cov_w2v = {}, {}, {}
for n in W2V_MODELS:
    m = te.get_embedder(n)
    E_w2v[n] = te.get_embeddings(m, df, force=FORCE)
    # ponowne kodowanie sprawdza cache i daje pokrycie słownika dla każdego zdania (w cache go nie ma)
    assert np.allclose(m.encode(df["text"].tolist()), E_w2v[n], atol=1e-5)
    emb_w2v[n], cov_w2v[n] = m, m.last_coverage
    print(f"OK  {n:10s} {E_w2v[n].shape}  {m.label}")

display(pd.DataFrame({
    "wymiar": [E_w2v[n].shape[1] for n in W2V_MODELS],
    "pokrycie słów: średnio": [f"{cov_w2v[n].mean():.1%}" for n in W2V_MODELS],
    "pokrycie słów: najmniej": [f"{cov_w2v[n].min():.0%}" for n in W2V_MODELS],
    "zdania bez znanego słowa": [int((cov_w2v[n] == 0).sum()) for n in W2V_MODELS],
}, index=pd.Index([W2V_TITLES[n] for n in W2V_MODELS], name="model")))

In [ ]:
for n in W2V_MODELS:
    print(f"\n=== {emb_w2v[n].label} ===")
    print(f"macierz: {E_w2v[n].shape[0]} zdań × {E_w2v[n].shape[1]} wymiarów, dtype {E_w2v[n].dtype}, "
          f"średnia norma wektora: {np.linalg.norm(E_w2v[n], axis=1).mean():.2f}")
    display(te.embedding_preview(df, E_w2v[n], n_rows=5, n_dims=8))
print("\npełne osadzenia: results/text_embeddings/embeddings_w2v.csv, embeddings_w2v_polemo.csv")

**Podobieństwo kosinusowe tych samych zdań co w 1.3** (po dwa z każdej klasy gold). Tak jak w fastText,
wszystkie zdania są do siebie podobne: poza przekątną cosinus wynosi średnio 0,66 (IPI PAN) i 0,61 (PolEmo),
od 0,39 do 0,84. Wysoki cosinus nie oznacza więc podobnego wydźwięku; sens ma tylko porównanie względne.

In [ ]:
sel = (df[df["iter"] == df["iter"].min()].groupby("gold", sort=False).head(2)).sort_values("gold").index  # jak w 1.3
for n in W2V_MODELS:
    te.similarity_heatmap(df, E_w2v[n], sel, f"Podobieństwo kosinusowe – {emb_w2v[n].label}").show()

## 2. Wizualizacja t-SNE z odniesieniem do etykiet [0,5 pkt]

Wykres jest interaktywny: **po najechaniu na punkt widać pełne zdanie oraz etykiety** (gold, A1, A2, final),
iterację i `sent_id`. Domyślnie punkty są kolorowane **uzgodnioną anotacją (`final`)**, a lista rozwijana nad wykresem przełącza kolorowanie między źródłami etykiet,
kliknięcie w legendę ukrywa klasę. Kółko = iteracja 1, romb = iteracja 2.
t-SNE: odległość kosinusowa, inicjalizacja PCA, `perplexity = min(30, (n−1)/3)`, stałe ziarno.
Wykresy zapisują się też jako HTML w `results/text_embeddings/`

In [ ]:
xy = {}
for n in E:
    xy[n] = te.tsne_2d(E[n])
    fig = te.make_tsne_figure(df, xy[n], f"t-SNE – {embedders[n].label}")
    fig.write_html(te.OUT_DIR / f"tsne_{n}.html", include_plotlyjs="cdn")
    fig.show()

**Jak czytać wykresy:** jeśli klasy wydźwięku tworzą oddzielne skupiska, osadzenie koduje sygnał przydatny
do klasyfikacji. Jeśli POZ i NEG są wymieszane, a skupiska mogą odpowiadać np. tematowi (hotel / lekarz /
szkoła / produkt) lub długości zdania, model koduje głównie temat.

### 2.1 t-SNE: uśrednione Word2Vec

Ta sama funkcja wykresu i te same ustawienia t-SNE co wyżej: **po najechaniu na punkt widać pełne zdanie, etykiety
gold, A1, A2 i final**, iterację i `sent_id`, a lista rozwijana nad wykresem zmienia kolorowanie. Pliki HTML tych
wykresów są już w `results/text_embeddings/tsne_w2v.html` i `tsne_w2v_polemo.html`
(`python -m src.text_embeddings --models w2v w2v_polemo --evaluate`), więc ta komórka niczego nie zapisuje.

In [ ]:
xy_w2v = {}
for n in W2V_MODELS:
    xy_w2v[n] = te.tsne_2d(E_w2v[n])
    te.make_tsne_figure(df, xy_w2v[n], f"t-SNE – {emb_w2v[n].label}").show()

**Co widać.** Klasy wydźwięku są wymieszane, tak jak dla fastText i USE: zdania POZ i NEG leżą obok siebie,
w tych samych skupiskach tematycznych. Wyróżnia się tylko część zdań NEU, które tworzą osobne skupisko, wyraźniej
w modelu PolEmo. Liczbowo (3.7.5): wśród 10 najbliższych sąsiadów zdania ta sama klasa `final` to 33% (IPI PAN)
i 36% (PolEmo), przy 30,5% losowo.

## 3. Analiza osadzeń i dyskusja [1 pkt]

odpowiedzi w 3.6

### 3.1 Pyt. 1 – zdania uznane za podobne, a różniące się znaczeniem
Pary zdań o przeciwnym wydźwięku (POZ vs NEG według gold) z najwyższym podobieństwem kosinusowym.

In [ ]:
sims = {n: te.cosine_matrix(E[n]) for n in E}
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.similar_but_different(df, sims[n], k=8))

### 3.2 Pyt. 2 – zdania podobne znaczeniowo, a uznane za odległe (inne słowa)
Dwie metody. **(a)** Pary, które jeden model uznaje za bliskie, a drugi za odległe (rangi percentylowe wszystkich par) – to kandydaci, trzeba je ocenić ręcznie. **(b)** Kontrolowane pary: parafrazy (inne słowa, ten sam sens) vs. kontrola niepowiązana.

In [ ]:
if len(real) >= 2:
    a, b = real[0], real[1]
else:
    a, b = list(E)[:2]
if len(E) >= 2:
    hi_a, hi_b = te.cross_model_disagreement(df, sims[a], sims[b], a, b, k=8)
    print(f"Blisko według {a}, daleko według {b}:"); display(hi_a)
    print(f"Blisko według {b}, daleko według {a}:"); display(hi_b)

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    probes = te.probe_table(embedders)      # modele ładują się z dysku; komunikaty ukryte
display(probes)
print("Średnie cos. wg kategorii:")
display(probes.groupby("kategoria", sort=False)[list(E)].mean().round(3))

Zdania w tabeli powyżej nie pochodzą z korpusu. to zestaw kontrolny napisany ręcznie (`PROBE_PAIRS` w `src/text_embeddings.py`), zrobiony po to, żeby sprawdzić zjawiska z pytań 1–3, które w 200 losowych zdaniach trudno znaleźć. Kolumna `kategoria` mówi, jaka jest relacja między a i b: `parafraza` (ten sam sens, inne słowa), `przeciwny sens, podobne słowa` („smaczne” / „niesmaczne”), `przeciwny sens, te same słowa` (te same słowa w innej kolejności), `ten sam sens, różna długość` oraz `kontrola: niepowiązane` (linia bazowa: ile wynosi cosinus „z niczego”). Liczby w kolumnach modeli to cosinus między wektorami obu zdań. Dla 12 par wnioski są ilustracją, a nie statystyką; dane z korpusu są w 3.1 i 3.2(a).

### 3.3 Pyt. 3 – różna długość, podobny sens
Kategoria „ten sam sens, różna długość” w tabeli powyżej (porównaj z parafrazami i z kontrolą niepowiązaną).

In [ ]:
len_pairs = probes[probes["kategoria"].str.contains("różna długość")].copy()
len_pairs["dł_a"] = len_pairs["zdanie_a"].str.split().str.len()
len_pairs["dł_b"] = len_pairs["zdanie_b"].str.split().str.len()
display(len_pairs[["zdanie_a", "zdanie_b", "dł_a", "dł_b"] + list(E)])

### 3.4 Pyt. 4 – długość zdania a wartości podobieństwa i „wyrazistość” wektora
Przedziały długości (kwantyle liczby tokenów): norma surowego wektora (dla uśrednianych wektorów słów to miara siły sygnału; dla USE norma jest zwykle ≈ stała, więc nie ma interpretacji), cosinus do centroidu korpusu i średnie podobieństwo do pozostałych zdań.

In [ ]:
for n in E:
    print(f"\n=== {embedders[n].label} ===")
    display(te.length_stats(df, E[n]))

fig = go.Figure()
for n in E:
    S = sims[n].copy(); np.fill_diagonal(S, np.nan)
    fig.add_trace(go.Scatter(x=df["n_tokens"], y=np.nanmean(S, axis=1), mode="markers", name=n,
                             opacity=0.6, text=df["text"], hovertemplate="%{text}<br>tokenów: %{x}, śr. cos: %{y:.3f}<extra>"+n+"</extra>"))
fig.update_layout(title="Długość zdania vs średnie podobieństwo do pozostałych zdań", template="plotly_white",
                  xaxis_title="liczba tokenów", yaxis_title="średni cosinus do innych zdań", height=520)
fig.show()

Zdania podzielono na 3 grupy wg liczby tokenów, tak aby w grupach było podobnie dużo zdań.

* `n` – liczba zdań w grupie.
* `norma_wektora` – średnia długość surowego wektora zdania. W fastText wektor zdania to średnia wektorów słów, więc w dłuższych zdaniach wektory częściowo się znoszą i norma maleje. W USE norma jest stała (1), więc ta kolumna nic nie mówi.
* `cos_do_centroidu` – podobieństwo zdania do środka (średniej) wszystkich zdań. Im wyżej, tym zdanie jest „bardziej przeciętne”.
* `srednie_cos_do_innych` – średnie podobieństwo zdania do wszystkich pozostałych.
* Ostatni wiersz – współczynnik Spearmana między liczbą tokenów a średnim cosinusem. Dodatni znak = im dłuższe zdanie, tym bardziej podobne do reszty.

Wniosek: w fastText długie zdania zbliżają się do średniego wektora, więc wysokie cosinusy mogą wynikać z długości, a nie z podobnego sensu.

### 3.5 Pyt. 5 – czy osadzenia nadają się do zadania (klasyfikacja wydźwięku zdania)?
Regresja logistyczna (walidacja krzyżowa warstwowa) na samych osadzeniach, czystość k najbliższych sąsiadów względem poziomu losowego i silhouette. **Punkt odniesienia dla accuracy to `baza_wiekszosciowa`** (udział najliczniejszej klasy, czyli wynik klasyfikatora zawsze wybierającego tę klasę); `losowy_wg_rozkladu` to losowanie zgodne z rozkładem klas, a `acc_minus_baza` mówi, o ile model go przekracza. Etykiety (gold, A1, A2, final) są liczone na zdaniach, które je mają. `final` to uzgodnione etykiety, więc to ją traktujemy jako główny punkt odniesienia; gold PolEmo i A1/A2 służą do porównania. Przy ~200 zdaniach wyniki są orientacyjne, nie ma tu przedziałów ufności.

In [ ]:
rows = []
for n in E:
    for col in te.label_columns(df):
        r = te.evaluate_embeddings(E[n], df[col]); r.update(model=n, etykiety=col); rows.append(r)
res = pd.DataFrame(rows).set_index(["model", "etykiety"])
display(res)

Klasyfikator (regresja logistyczna) przewiduje wydźwięk zdania z samego wektora zdania, czyli sprawdzamy jakość osadzeń, a nie samego klasyfikatora. Wiersze to model osadzeń (`fasttext`, `use`, `tfidf` jako punkt odniesienia) × zestaw etykiet, które traktujemy jako prawdę (`final`, `gold`, `A1`, `A2`). Wynik jest z 5-krotnej walidacji krzyżowej, więc klasyfikator nie ocenia zdań, na których się uczył.

* `accuracy` – odsetek poprawnych przewidywań. Porównuj ją z `baza_wiekszosciowa` (wynik klasyfikatora, który zawsze wybiera najliczniejszą klasę); `acc_minus_baza` to różnica. Przy `final` i A1/A2 klasa NEU jest najliczniejsza, więc same accuracy bywa mylące.
* `macro_F1` – średnia F1 po klasach, każda klasa liczy się tak samo, więc ignorowanie rzadkich klas (AMB) mocno obniża wynik.
* `kNN10_czystosc` – średni odsetek 10 najbliższych sąsiadów zdania (wg cosinusa) z tą samą etykietą. Nie używa klasyfikatora, mierzy samą geometrię osadzenia. Odnosi się do `losowy_wg_rozkladu` (losowanie zgodne z rozkładem klas).
* `silhouette_cos` – zwykle miara klasteryzacji, ale działa dla dowolnego podziału na grupy, także dla etykiet. Pokazuje, czy zdania jednej klasy leżą bliżej siebie niż zdań innych klas (od −1 do 1). Wartości ≤ 0 to brak oddzielnych skupisk, zgodnie z t-SNE.
* Różnice między `gold` a `final`/A1/A2: `gold` jest zbalansowane (po 50 zdań na klasę), a w `final` i A1/A2 dominuje NEU, więc accuracy jest wyższa głównie dlatego, że łatwiej trafić w NEU.

Macierz pomyłek (wiersze = uzgodniona etykieta `final`, kolumny = predykcja klasyfikatora na samych
osadzeniach) i precyzja / czułość / F1 dla każdej klasy. Pokazuje, czy problem leży w klasach „środkowych” (NEU, AMB), czy też
osadzenia mylą nawet POZ z NEG.

In [ ]:
main_col = "final" if "final" in te.label_columns(df) else "gold"
print("etykiety:", main_col)
for n in E:
    cm, rep = te.confusion_report(E[n], df[main_col])
    print(f"\n=== {embedders[n].label} ===")
    display(cm); display(rep)

Wiersz to prawdziwa etykieta (`final`), kolumna to to, co przewidział klasyfikator. Liczby na przekątnej to trafienia. Jeśli prawie wszystko ląduje w kolumnie NEU, klasyfikator „ucieka” do najliczniejszej klasy. Zwróć uwagę na `recall` NEG i AMB: bliskie 0 oznacza, że osadzenia prawie nie odróżniają tych klas od NEU.

### 3.6 Odpowiedzi na pytania pomocnicze (do uzupełnienia po uruchomieniu)

Każda odpowiedź ma opierać się na konkretnych przykładach i liczbach z sekcji 3.1–3.5.

**1. Czy są zdania, które model uznał za podobne, ale w rzeczywistości różnią się znaczeniem?**
_Odpowiedź:_ …

**2. Czy są zdania semantycznie podobne, ale model uznał je za odległe, bo użyto innych słów?**
_Odpowiedź:_ … 

**3. Czy zdania o różnej długości, ale podobnym sensie, są traktowane przez model podobnie?**
_Odpowiedź:_ … 

**4. Czy zauważasz różnicę w wartościach podobieństwa między krótkimi a długimi zdaniami? Czy długość wpływa na „wyrazistość” wektora?**
_Odpowiedź:_ … 

**5. Czy osadzenia zdań nadają się do wybranego zadania (anotacja wydźwięku)?**
_Odpowiedź:_ … 

### 3.7 Word2Vec: analiza osadzeń (pyt. 1–5)

Te same funkcje, pary kontrolne i miary co w 3.1–3.5, tym razem dla uśrednionego Word2Vec. Dochodzą dwie miary:
* pyt. 1: udział par POZ–NEG wśród 1% najbardziej podobnych par zdań,
* pyt. 2: zależność podobieństwa od wspólnych słów (Spearman: kosinus wobec miary Jaccarda zbiorów słów).

W tabelach porównawczych fastText, USE i TF-IDF to wyniki zespołu z sekcji 3.2–3.5, wczytane z tabel sprawozdania
(`report/generated/emb_*.tex`, funkcje `team_*` w `src/text_analysis.py`). USE z TF Hub nie działa w Windows,
więc tych modeli nie liczymy tu od nowa. Odpowiedzi na pytania są w 3.7.6.

#### 3.7.1 Pyt. 1: zdania uznane za podobne, a różne znaczeniowo
Najbardziej podobne pary zdań o przeciwnym wydźwięku (POZ i NEG według etykiety `final`) oraz udział par POZ–NEG
wśród 1% najbardziej podobnych par zdań, w porównaniu z udziałem wśród wszystkich par.

In [ ]:
from src import text_analysis as ta

sims_w2v = {n: te.cosine_matrix(E_w2v[n]) for n in W2V_MODELS}
for n in W2V_MODELS:
    print(f"\n=== {emb_w2v[n].label} ===")
    display(ta.top_pairs(df, sims_w2v[n], k=8, opposite_only=True))

top_share = pd.DataFrame.from_dict(
    {W2V_TITLES[n]: ta.opposite_share_top(sims_w2v[n], df["final"].to_numpy()) for n in W2V_MODELS}, orient="index")
display(top_share.rename(columns={
    "n_top": "par w górnym 1%", "top_opposite": "POZ–NEG: górny 1%", "all_opposite": "POZ–NEG: wszystkie pary",
    "top_same": "ta sama etykieta: górny 1%", "all_same": "ta sama etykieta: wszystkie pary"}).round(3))

#### 3.7.2 Pyt. 2: zdania podobne znaczeniowo, a uznane za odległe (inne słowa)
**(a)** Jak bardzo podobieństwo zależy od wspólnych słów: korelacja Spearmana kosinusa z miarą Jaccarda zbiorów słów
(wszystkie pary zdań) i średni kosinus w grupach par. **(b)** Pary, które jeden model uznaje za bliskie, a drugi
za odległe (rangi percentylowe, jak w 3.2; tu IPI PAN wobec PolEmo). To kandydaci do ręcznej oceny.
**(c)** Pary kontrolne z 3.2 dla Word2Vec i średnie dla wszystkich pięciu modeli.

In [ ]:
J_words = ta.jaccard_matrix(df["text"].tolist())
lex_w2v = {n: ta.lexical_dependence(sims_w2v[n], J_words) for n in W2V_MODELS}
display(pd.DataFrame.from_dict(
    {W2V_TITLES[n]: {"Spearman (cos vs Jaccard)": q["spearman_cos_jaccard"],
                     **{f"śr. cos: {k}": v for k, v in q["mean_cos"].items()}} for n, q in lex_w2v.items()},
    orient="index").round(3))
print("liczba par w grupach:", lex_w2v[W2V_MODELS[0]]["n_pairs"])

close_ipi, close_pol = te.cross_model_disagreement(df, sims_w2v["w2v"], sims_w2v["w2v_polemo"],
                                                   "w2v", "w2v_polemo", k=8)
print("\nBlisko według IPI PAN, daleko według PolEmo:"); display(close_ipi)
print("Blisko według PolEmo, daleko według IPI PAN:"); display(close_pol)

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    probes_w2v = te.probe_table(emb_w2v)
display(probes_w2v)
print("Średnie cos. wg kategorii (fastText, USE, TF-IDF: wyniki zespołu z 3.2):")
display(probes_w2v.groupby("kategoria", sort=False)[W2V_MODELS].mean().rename(columns=W2V_TITLES)
        .join(ta.team_probe_means()).round(3))

#### 3.7.3 Pyt. 3: różna długość, podobny sens
Kategoria „ten sam sens, różna długość” z par kontrolnych (porównaj z parafrazami i z kontrolą niepowiązaną
w tabeli powyżej).

In [ ]:
len_pairs_w2v = probes_w2v[probes_w2v["kategoria"].str.contains("różna długość")].copy()
len_pairs_w2v["dł_a"] = len_pairs_w2v["zdanie_a"].str.split().str.len()
len_pairs_w2v["dł_b"] = len_pairs_w2v["zdanie_b"].str.split().str.len()
display(len_pairs_w2v[["zdanie_a", "zdanie_b", "dł_a", "dł_b"] + W2V_MODELS])

#### 3.7.4 Pyt. 4: długość zdania a podobieństwo i „wyrazistość” wektora
Te same kolumny co w 3.4 (opis pod tabelami w 3.4). Wykres jest interaktywny: po najechaniu na punkt widać zdanie
i jego etykietę `final`.

In [ ]:
for n in W2V_MODELS:
    print(f"\n=== {emb_w2v[n].label} ===")
    display(te.length_stats(df, E_w2v[n]))
team_len = ta.team_length()
print("Spearman(n_tokens, śr. cos) w modelach zespołu (3.4):")
display(team_len.loc[team_len["grupa"] == "Spearman", ["model", "spearman"]].set_index("model"))

fig = go.Figure()
for n in W2V_MODELS:
    S = sims_w2v[n].copy(); np.fill_diagonal(S, np.nan)
    fig.add_trace(go.Scatter(x=df["n_tokens"], y=np.nanmean(S, axis=1), mode="markers", name=W2V_TITLES[n],
                             opacity=0.6, text=df["text"].str.wrap(70).str.replace("\n", "<br>"),
                             customdata=df["final"].fillna("brak"),
                             hovertemplate="%{text}<br>final: %{customdata} | tokenów: %{x}, śr. cos: %{y:.3f}"
                                           "<extra>" + W2V_TITLES[n] + "</extra>"))
fig.update_layout(title="Długość zdania vs średnie podobieństwo do pozostałych zdań – uśrednione Word2Vec",
                  template="plotly_white", xaxis_title="liczba tokenów", yaxis_title="średni cosinus do innych zdań",
                  height=520)
fig.show()

#### 3.7.5 Pyt. 5: czy osadzenia nadają się do zadania (wydźwięk zdania)?
Te same miary co w 3.5: regresja logistyczna na samych osadzeniach (5-krotna walidacja krzyżowa warstwowa), czystość
10 najbliższych sąsiadów wobec poziomu losowego i silhouette, dla każdego źródła etykiet. Dalej porównanie pięciu
modeli, macierze pomyłek i czułość dla każdej klasy (etykiety `final`).

In [ ]:
TEAM_NAMES = {"fasttext": "fastText", "use": "USE", "tfidf": "TF-IDF"}
ev_w2v = pd.concat({n: te.evaluation_table(df, E_w2v[n]).set_index("etykiety") for n in W2V_MODELS}, names=["model"])
display(ev_w2v)

cols = {"accuracy": "accuracy", "baza_wiekszosciowa": "baza", "macro_F1": "macro_F1",
        "kNN10_czystosc": "kNN10", "losowy_wg_rozkladu": "losowy", "silhouette_cos": "silhouette"}
ours = ev_w2v.xs("final", level="etykiety")[list(cols)].rename(columns=cols, index=W2V_TITLES)
team_cls = ta.team_classif("final").set_index("model").rename(index=TEAM_NAMES)
print("Etykiety final, pięć modeli (fastText, USE, TF-IDF: wyniki zespołu z 3.5):")
display(pd.concat([ours, team_cls]))

In [ ]:
recall_w2v = {}
for n in W2V_MODELS:
    cm, rep = te.confusion_report(E_w2v[n], df["final"])
    print(f"\n=== {emb_w2v[n].label} ===")
    display(cm); display(rep)
    recall_w2v[W2V_TITLES[n]] = rep.loc[te.config.SENT_LABELS, "recall"]
print("Czułość (recall) dla każdej klasy, etykiety final (fastText, USE, TF-IDF: wyniki zespołu z 3.5):")
display(pd.concat([pd.DataFrame(recall_w2v).T, ta.team_recall().rename(index=TEAM_NAMES)]).round(3))

#### 3.7.6 Odpowiedzi na pytania pomocnicze (Word2Vec)

Odpowiedzi dla uśrednionego Word2Vec na podstawie 3.7.1–3.7.5. Przy dwóch liczbach pierwsza dotyczy modelu IPI PAN,
druga PolEmo. Opierają się na 200 zdaniach i 12 parach kontrolnych, więc są ilustracją, a nie statystyką.
Ten sam tekst jest w `results/text_embeddings/odpowiedzi_w2v.md`, a w skrócie w sprawozdaniu.

**1. Czy są zdania, które model uznał za podobne, a różnią się znaczeniem?**

**Tak, i to systematycznie.**
- **Pary kontrolne.** Pary o przeciwnym sensie i podobnych słowach są bliższe niż prawdziwe parafrazy:
  - średnio 0,882 wobec 0,710 (IPI PAN) i 0,877 wobec 0,700 (PolEmo),
  - „Obsługa była bardzo miła” i „…bardzo niemiła”: 0,97 / 0,96,
  - „Jedzenie było smaczne” i „…niesmaczne”: 0,93 / 0,97,
  - „Polecam ten hotel każdemu” i „Nie polecam tego hotelu nikomu”: 0,74 / 0,70, czyli tyle co parafrazy,
  - zamiana ocen („obsługa świetna, pokój fatalny” i odwrotnie): 0,95 / 0,93. Średnia nie widzi kolejności słów, więc to prawie ten sam wektor.

  Zespół zaobserwował to samo w fastText (0,859 wobec 0,746) i USE.
- **Pary z korpusu.** Wśród 1% najbardziej podobnych par (199 par):
  - pary POZ–NEG stanowią 15,1% (IPI PAN) i 16,1% (PolEmo), więcej niż wśród wszystkich par (12,5%),
  - pary o tej samej etykiecie stanowią 29–31%, tyle co przypadkiem (30,2%).

  Bliskość wektorów w ogóle nie odróżnia więc wydźwięku. Najbliższe pary POZ–NEG (cos 0,90–0,92) to długie opinie o lekarzach, np. „…jest to wyjątkowy lekarz…” (POZ) i „Myślę, że lekarz powinien dokładnie wysłuchać pacjenta…” (NEG): wspólny temat i słownictwo, przeciwna ocena.
- **Przyczyna.** Średnia wektorów słów koduje temat, a nie wydźwięk:
  - słowa o przeciwnym wydźwięku leżą w word2vec blisko siebie (cos(dobry, zły) = 0,73, część Word Embeddings),
  - „nie” to tylko jedno słowo w średniej, które nie zmienia wektorów pozostałych słów.

**2. Czy są zdania podobne znaczeniowo, a uznane za odległe, bo użyto innych słów?**

**Częściowo.**
- **Parafrazy kontra zdania niepowiązane.** Parafrazy dostają 0,71 / 0,70, wyraźnie więcej niż pary niepowiązane (0,43 / 0,43). Ten kontrast jest większy niż w fastText (0,746 wobec 0,506), bo word2vec zna synonimy (miła ≈ uprzejma).
- **Zależność od wspólnych słów.**
  - Korelacja Spearmana podobieństwa ze wspólnymi słowami (Jaccard zbiorów słów) wynosi 0,45 / 0,50.
  - Pary bez wspólnych słów mają średnio 0,62 / 0,57, a pary o Jaccardzie > 0,1 mają 0,77 / 0,74.
  - Inne słowa obniżają więc podobieństwo, choć nie do zera, bo wszystkie zdania leżą blisko średniego wektora (pyt. 4).
- **Przykład odległości z powodu innych form słów.** Zdania „Ogólnie Czystość i pokoje na 10, atmosfera na 3.” i „Czysto, schludnie, bardzo wygodne łóżka.”:
  - w IPI PAN leżą daleko od siebie (ranga 0,15 wśród wszystkich par),
  - w PolEmo blisko (0,94).

  Model form traktuje „czystość” i „czysto” jak różne słowa, a model uczony na recenzjach łączy je przez kontekst.
- **Przykład odwrotny.** IPI PAN zbliża zdania o czasie („Używam jej od ok 2 lat.” i „A było to 2 miesiące temu.”, ranga 0,95), których PolEmo nie łączy (0,14). Czy para jest naprawdę podobna znaczeniowo, trzeba ocenić ręcznie, bo modele różnią się tym, które relacje między słowami znają.

**3. Czy zdania o różnej długości i podobnym sensie są traktowane podobnie?**

**Nie w pełni.**
- **Średnie wartości.** Pary „krótkie zdanie i jego długa wersja” mają średnio 0,58 (IPI PAN) i 0,66 (PolEmo). To mniej niż parafrazy (0,71 / 0,70), choć więcej niż zdania niepowiązane (0,43).
- **Przykład.** „Polecam.” i długa pochwała hotelu mają 0,50 / 0,60.
- **Przyczyna.** Krótkie zdanie to wektor jednego lub kilku słów. Długie to średnia wielu słów (hotel, znajomi, rodzina, tydzień), które odciągają wektor od oceny i zbliżają go do przeciętnego zdania korpusu.
- **Różnica między modelami.** Model PolEmo radzi sobie trochę lepiej, bo jego słowa niosą kontekst recenzji („polecam” leży blisko słów o hotelach).
- **Porównanie z innymi modelami.** fastText daje 0,60, a USE tylko 0,19.

**4. Czy długość zdania wpływa na podobieństwo i „wyrazistość” wektora?**

**Tak, wyraźnie, i silniej niż w fastText.** Od zdań krótkich (5–11 tokenów) do długich (21–36):
- **norma wektora maleje:** 2,13 → 1,81 (IPI PAN) i 2,35 → 1,91 (PolEmo),
- **podobieństwo do średniego wektora korpusu rośnie:** 0,747 → 0,885 i 0,718 → 0,847,
- **średnie podobieństwo do pozostałych zdań rośnie:** 0,61 → 0,73,
- **korelacja Spearmana** długości ze średnim podobieństwem: 0,77 / 0,76, wobec 0,59 w fastText i 0,32 w USE.

**Przyczyna.** W średniej wielu słów ich wkłady częściowo się znoszą, więc wektor długiego zdania jest krótszy i bliższy „przeciętnemu zdaniu”, czyli mniej wyrazisty.

**Konsekwencje:**
- najbardziej podobne pary w korpusie to głównie długie zdania (pyt. 1),
- wysokie podobieństwo może wynikać z samej długości, a nie z sensu,
- pomogłoby ważenie słów (rzadsze ważniejsze) i usunięcie wspólnej składowej wszystkich zdań, jak w metodzie SIF.

**5. Czy osadzenia nadają się do wybranego zadania (wydźwięk zdania)?**

**Nie samodzielnie.**
- **Wyniki klasyfikatora.** Regresja logistyczna na samych osadzeniach (etykiety `final`):

  | Model | Trafność | Baza większościowa | Macro-F1 | Czystość 10 sąsiadów (losowo 0,305) | Sylwetka |
  |---|---|---|---|---|---|
  | IPI PAN | 0,505 | 0,415 | 0,31 | 0,33 | ujemna |
  | PolEmo | 0,540 | 0,415 | 0,35 | 0,36 | ujemna |

  Przewaga nad bazą większościową to +0,09 i +0,13.
- **Czułość dla poszczególnych klas.** Klasyfikator wybiera głównie NEU (czułość 0,89). POZ rozpoznaje w 44% / 54% przypadków, NEG prawie wcale (7% / 11%), AMB w ogóle. To zgodne z pyt. 1: negacja i antonimy nie zmieniają średniej.
- **Porównanie z innymi modelami.** Model PolEmo wypada najlepiej z pięciu porównanych (USE 0,520, fastText 0,505, TF-IDF 0,490). Uczono go jednak na tym samym korpusie recenzji, więc ma przewagę „dziedzinową”.
- **Do czego się nadają.** Najwyżej jako cecha pomocnicza (temat, dziedzina recenzji).
- **Co jest potrzebne zamiast tego:**
  - model uwzględniający kolejność słów i negację, np. dostrojony transformer (HerBERT),
  - albo osadzenia uczone z etykietami wydźwięku.

**Właściwości badanych modeli: podsumowanie**

| | Word2Vec IPI PAN (ogólny) | Word2Vec PolEmo (dziedzinowy) |
|---|---|---|
| co koduje wektor zdania | temat i słownictwo; relacje słownikowe (synonimy, wyrażenia czasu) | temat i skojarzenia z recenzji (czystość, łazienka, lekarz) |
| wydźwięk | prawie niewidoczny (antonimy blisko, negacja ginie w średniej) | nieco lepszy (najwyższa trafność), nadal słaby dla NEG |
| długość zdania | silny efekt: długie zdania zbliżają się do średniej korpusu | to samo |
| słownik | 96,7% słów w zdaniach ma wektor | 91,0%; słowa rzadkie (< 3 wystąpienia) pominięte |